# Custom Derivatives and Advanced Differentiation

This notebook goes deeper into JAX's differentiation system — custom gradient rules, Hessians, and practical gradient patterns used in real training.

### What we've covered so far

In notebook 50, we learned `jax.grad` for basic derivatives. Now we'll learn:
- **Forward-mode** vs **reverse-mode** differentiation
- How to **define custom gradients** for functions JAX can't differentiate
- **Hessians** (second-order derivatives as a matrix)
- Practical patterns: **gradient clipping**, **auxiliary data**, **gradient accumulation**

In [1]:
# Import JAX libraries
import jax
import jax.numpy as jnp
from functools import partial

### ✅ Exercise 81 — Forward-Mode vs Reverse-Mode

JAX supports two differentiation modes:

| Mode | Function | Best for |
|---|---|---|
| **Reverse** (`grad`) | Many inputs → 1 output | Neural networks (many weights, scalar loss) |
| **Forward** (`jvp`) | 1 input → many outputs | Jacobian columns, directional derivatives |

`jvp` stands for **Jacobian-Vector Product** — it computes how a small change in input propagates *forward* through the function.

```python
primals_out, tangents_out = jax.jvp(f, (primals,), (tangents,))
```
- `primals` = the point to evaluate at
- `tangents` = the direction to differentiate along (like a "nudge")
- Returns: function value + derivative in that direction

In [2]:
# Forward-mode: compute f(x) and f'(x) in one pass
f = lambda x: jnp.sin(x) * x**2

x = jnp.float32(2.0)
tangent = jnp.float32(1.0)  # direction = 1.0 means "give me the full derivative"

primal_out, tangent_out = jax.jvp(f, (x,), (tangent,))
print(f"f(2.0)  = {primal_out:.4f}")
print(f"f'(2.0) = {tangent_out:.4f}")

# Compare with reverse-mode
grad_val = jax.grad(f)(x)
print(f"grad    = {grad_val:.4f}")  # same answer, different algorithm

f(2.0)  = 3.6372
f'(2.0) = 1.9726
grad    = 1.9726


### ✅ Exercise 82 — Hessian Matrix

The **Hessian** is a matrix of all second derivatives. For a function f(x₁, x₂):

```
H = [[∂²f/∂x₁², ∂²f/∂x₁∂x₂],
     [∂²f/∂x₂∂x₁, ∂²f/∂x₂²]]
```

It tells you about the **curvature** of the function — used in second-order optimizers like Newton's method.

JAX computes it by applying `jacobian` to `grad` (Jacobian of the gradient).

In [3]:
# f(x) = x₁² + 3*x₁*x₂ + x₂³
def f(x):
    return x[0]**2 + 3*x[0]*x[1] + x[1]**3

x = jnp.array([1.0, 2.0])

# Gradient (first derivatives)
grad_f = jax.grad(f)(x)
print(f"Gradient: {grad_f}")  # [2*x₁ + 3*x₂, 3*x₁ + 3*x₂²] = [8, 15]

# Hessian (second derivatives)
hessian_f = jax.hessian(f)(x)
print(f"Hessian:\n{hessian_f}")
# [[2,  3],    ∂²f/∂x₁² = 2,   ∂²f/∂x₁∂x₂ = 3
#  [3, 12]]    ∂²f/∂x₂∂x₁ = 3,  ∂²f/∂x₂² = 6*x₂ = 12

Gradient: [ 8. 15.]
Hessian:
[[ 2.  3.]
 [ 3. 12.]]


### ✅ Exercise 83 — Hessian-Vector Product (Efficient Second-Order)

Computing the full Hessian is expensive (O(n²) for n parameters). In practice, we often only need the **Hessian-vector product** Hv, which can be computed in O(n) time.

The trick: use forward-over-reverse differentiation.
- First, `grad` gives us the gradient function (reverse mode)
- Then, `jvp` computes a directional derivative of the gradient (forward mode)
- Result: Hv without ever forming the full Hessian

In [4]:
def f(x):
    return x[0]**2 + 3*x[0]*x[1] + x[1]**3

x = jnp.array([1.0, 2.0])
v = jnp.array([1.0, 0.0])  # direction vector

# Hessian-vector product: H @ v
# jvp of grad gives us one column/row of the Hessian
_, hvp = jax.jvp(jax.grad(f), (x,), (v,))
print(f"Hv (forward-over-reverse): {hvp}")

# Verify: H @ v using full Hessian
H = jax.hessian(f)(x)
print(f"Hv (full Hessian @ v):     {H @ v}")

Hv (forward-over-reverse): [2. 3.]
Hv (full Hessian @ v):     [2. 3.]


### ✅ Exercise 84 — value_and_grad with Auxiliary Data

In real training, your loss function often computes useful things besides the loss — like accuracy, logits, or intermediate activations. `has_aux=True` lets you return these **without** differentiating through them.

```python
# Without has_aux: grad expects f to return a scalar
# With has_aux:    f returns (scalar, anything) — grad ignores the second part
```

In [5]:
def loss_with_metrics(params, x, y_true):
    pred = params['w'] * x + params['b']
    loss = jnp.mean((pred - y_true) ** 2)
    # Auxiliary data: things we want to see but not differentiate
    metrics = {
        'predictions': pred,
        'max_error': jnp.max(jnp.abs(pred - y_true)),
    }
    return loss, metrics  # (differentiable, not differentiable)

params = {'w': jnp.float32(1.0), 'b': jnp.float32(0.0)}
x = jnp.array([1.0, 2.0, 3.0])
y = jnp.array([2.0, 4.0, 6.0])

# has_aux=True: "my function returns (loss, extras)"
(loss, metrics), grads = jax.value_and_grad(loss_with_metrics, has_aux=True)(params, x, y)

print(f"Loss:      {loss:.4f}")
print(f"Max error: {metrics['max_error']:.4f}")
print(f"Grads:     w={grads['w']:.4f}, b={grads['b']:.4f}")

Loss:      4.6667
Max error: 3.0000
Grads:     w=-9.3333, b=-4.0000


### ✅ Exercise 85 — Gradient Clipping

Large gradients can destabilize training. **Gradient clipping** scales down the entire gradient vector if its norm exceeds a threshold, preserving direction while limiting magnitude.

```
If ‖g‖ > max_norm:
    g = g × (max_norm / ‖g‖)
```

In [6]:
def clip_grads_by_norm(grads, max_norm):
    """Clip gradient pytree by global norm."""
    # Compute global norm across all leaves
    leaves = jax.tree.leaves(grads)
    global_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in leaves))
    # Scale factor: 1.0 if within limit, smaller if exceeding
    scale = jnp.minimum(1.0, max_norm / global_norm)
    return jax.tree.map(lambda g: g * scale, grads)

# Simulate large gradients
grads = {
    'W': jnp.array([[10.0, -20.0], [15.0, -5.0]]),
    'b': jnp.array([8.0, -12.0]),
}

leaves = jax.tree.leaves(grads)
original_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in leaves))
print(f"Original norm: {original_norm:.2f}")

clipped = clip_grads_by_norm(grads, max_norm=5.0)
clipped_leaves = jax.tree.leaves(clipped)
clipped_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in clipped_leaves))
print(f"Clipped norm:  {clipped_norm:.2f}")
print(f"Clipped W:\n{clipped['W']}")

Original norm: 30.95
Clipped norm:  5.00
Clipped W:
[[ 1.6154267 -3.2308533]
 [ 2.42314   -0.8077133]]


### ✅ Exercise 86 — custom_vjp (Custom Reverse-Mode Gradient)

Sometimes JAX can't differentiate a function (e.g., it calls external code, or you want a numerically stable gradient). `custom_vjp` lets you **define your own backward pass**.

Two parts:
1. **Forward (`fwd`)**: compute the output + save values needed for the backward pass ("residuals")
2. **Backward (`bwd`)**: given the saved residuals and incoming gradient, return the gradient w.r.t. inputs

Think of it as telling JAX: "I know the math better than automatic differentiation here."

In [7]:
# Example: numerically stable log(1 + exp(x)) = softplus
# The naive gradient exp(x)/(1+exp(x)) overflows for large x
# But mathematically it's just sigmoid(x)

@jax.custom_vjp
def safe_softplus(x):
    return jnp.log(1.0 + jnp.exp(x))

def safe_softplus_fwd(x):
    out = safe_softplus(x)
    return out, x  # save x as residual for backward pass

def safe_softplus_bwd(x, g):
    # g is the incoming gradient (from downstream)
    # gradient of softplus = sigmoid(x) — numerically stable!
    return (g * jax.nn.sigmoid(x),)

safe_softplus.defvjp(safe_softplus_fwd, safe_softplus_bwd)

# Test
x = jnp.float32(100.0)  # large value where naive grad would overflow
print(f"softplus(100)      = {safe_softplus(x):.4f}")
print(f"grad softplus(100) = {jax.grad(safe_softplus)(x):.4f}")  # → ~1.0 (sigmoid(100) ≈ 1)

softplus(100)      = inf
grad softplus(100) = 1.0000


### ✅ Exercise 87 — custom_jvp (Custom Forward-Mode Gradient)

`custom_jvp` is the forward-mode counterpart. You define how tangents (directional derivatives) propagate through your function.

Simpler than `custom_vjp` when you just want to override the derivative rule.

In [8]:
# Example: define a function whose derivative we override
# f(x) = x², but we pretend f'(x) = 1 (identity gradient)
# This is called the "straight-through estimator" — used for quantization in ML

@jax.custom_jvp
def straight_through_square(x):
    return x ** 2

@straight_through_square.defjvp
def ste_jvp(primals, tangents):
    (x,) = primals
    (x_dot,) = tangents
    primal_out = straight_through_square(x)
    tangent_out = x_dot  # pretend derivative is 1 (straight-through)
    return primal_out, tangent_out

x = jnp.float32(3.0)
print(f"f(3)            = {straight_through_square(x):.1f}")  # → 9.0
print(f"custom f'(3)    = {jax.grad(straight_through_square)(x):.1f}")  # → 1.0 (not 6.0!)
print(f"real d/dx(x²)   = {jax.grad(lambda x: x**2)(x):.1f}")          # → 6.0

f(3)            = 9.0
custom f'(3)    = 1.0
real d/dx(x²)   = 6.0


### ✅ Exercise 88 — Gradient Accumulation

When your batch is too large to fit in memory, you process it in **mini-chunks** and accumulate gradients. The result is mathematically identical to processing the full batch.

```
Full batch gradient ≈ average of chunk gradients
```

In [9]:
def mse_loss(params, x, y):
    pred = params['w'] * x + params['b']
    return jnp.mean((pred - y) ** 2)

# Full dataset
key = jax.random.PRNGKey(0)
X = jax.random.normal(key, (64,))
Y = 3.0 * X + 1.0
params = {'w': jnp.float32(0.0), 'b': jnp.float32(0.0)}

# Method 1: Full batch gradient
full_grads = jax.grad(mse_loss)(params, X, Y)

# Method 2: Accumulate over 4 chunks of 16
n_chunks = 4
chunk_size = 16
acc_grads = jax.tree.map(jnp.zeros_like, params)

for i in range(n_chunks):
    chunk_x = X[i*chunk_size:(i+1)*chunk_size]
    chunk_y = Y[i*chunk_size:(i+1)*chunk_size]
    chunk_grads = jax.grad(mse_loss)(params, chunk_x, chunk_y)
    acc_grads = jax.tree.map(lambda a, g: a + g, acc_grads, chunk_grads)

# Average the accumulated gradients
acc_grads = jax.tree.map(lambda g: g / n_chunks, acc_grads)

print(f"Full batch grad:   w={full_grads['w']:.4f}, b={full_grads['b']:.4f}")
print(f"Accumulated grad:  w={acc_grads['w']:.4f}, b={acc_grads['b']:.4f}")

W0309 14:05:19.617538 12391393 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


Full batch grad:   w=-5.5999, b=-2.1878
Accumulated grad:  w=-5.5999, b=-2.1878


### ✅ Exercise 89 — Gradient of a Loss w.r.t. Inputs (Saliency)

Usually we differentiate w.r.t. **parameters** (for training). But differentiating w.r.t. **inputs** tells us which input features the model is most sensitive to. This is the basis of:
- **Saliency maps** (which pixels matter most)
- **Adversarial attacks** (which pixels to perturb)
- **Feature attribution**

In [10]:
# Simple model
def model(params, x):
    return jnp.sum(jnp.tanh(params['W'] @ x + params['b']))

key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)
params = {
    'W': jax.random.normal(k1, (4, 3)),
    'b': jax.random.normal(k2, (4,)),
}
x = jnp.array([1.0, 0.5, -1.0])

# Gradient w.r.t. params (for training) — argnums=0
param_grads = jax.grad(model, argnums=0)(params, x)
print(f"Param grad W shape: {param_grads['W'].shape}")

# Gradient w.r.t. input (for saliency) — argnums=1
input_grad = jax.grad(model, argnums=1)(params, x)
print(f"Input grad: {input_grad}")
print(f"Most sensitive feature: x[{jnp.argmax(jnp.abs(input_grad))}]")

Param grad W shape: (4, 3)
Input grad: [ 1.0021521  -0.16603808  0.2204163 ]
Most sensitive feature: x[0]


### ✅ Exercise 90 — Full Training Loop with All Patterns

Putting it all together: a JIT-compiled training loop with `value_and_grad`, auxiliary metrics, gradient clipping, and pytree parameter updates.

In [11]:
import jax.lax as lax

def init_params(key, layer_sizes):
    params = []
    for i in range(len(layer_sizes) - 1):
        key, k1 = jax.random.split(key)
        W = jax.random.normal(k1, (layer_sizes[i+1], layer_sizes[i])) * 0.01
        b = jnp.zeros(layer_sizes[i+1])
        params.append({'W': W, 'b': b})
    return params

def forward(params, x):
    for layer in params[:-1]:
        x = jnp.tanh(layer['W'] @ x + layer['b'])
    last = params[-1]
    return last['W'] @ x + last['b']  # no activation on last layer

def loss_fn(params, x, y):
    pred = forward(params, x)
    loss = jnp.mean((pred - y) ** 2)
    return loss, pred  # (loss, auxiliary)

def train_step(carry, _):
    params, x, y = carry
    (loss, pred), grads = jax.value_and_grad(loss_fn, has_aux=True)(params, x, y)
    # Clip gradients
    grads = clip_grads_by_norm(grads, max_norm=1.0)
    # Update
    params = jax.tree.map(lambda p, g: p - 0.01 * g, params, grads)
    return (params, x, y), loss

# Setup
key = jax.random.PRNGKey(0)
params = init_params(key, [3, 16, 8, 2])
x = jnp.array([1.0, 0.5, -1.0])
y = jnp.array([1.0, 0.0])

# Train with scan (JIT-compiled loop)
(final_state, _, _), losses = jax.jit(
    lambda carry: lax.scan(train_step, carry, None, length=500)
)((params, x, y))

print(f"Initial loss: {losses[0]:.6f}")
print(f"Final loss:   {losses[-1]:.6f}")
print(f"Prediction:   {forward(final_state, x)}")
print(f"Target:       {y}")

Initial loss: 0.500009
Final loss:   0.000021
Prediction:   [ 9.9352646e-01 -6.2084728e-06]
Target:       [1. 0.]


## Conclusion

### Differentiation Toolbox Summary

| Tool | What it does | When to use |
|---|---|---|
| `jax.grad` | Reverse-mode gradient | Most training (many params → scalar loss) |
| `jax.jvp` | Forward-mode derivative | Directional derivatives, few inputs |
| `jax.hessian` | Full second-derivative matrix | Small problems, analysis |
| `jvp(grad(f))` | Hessian-vector product | Large-scale second-order methods |
| `value_and_grad(has_aux)` | Loss + metrics + gradients | Real training loops |
| `custom_vjp` | Custom backward pass | Numerical stability, external code |
| `custom_jvp` | Custom forward derivative | Straight-through estimators |
| `grad(f, argnums=1)` | Gradient w.r.t. inputs | Saliency, adversarial attacks |

These tools, combined with `jit`, `vmap`, and pytrees from earlier notebooks, form the complete JAX differentiation stack.